## OpenMeteo API

In [ ]:
import openmeteo_requests
import pandas as pd
import requests_cache
from retry_requests import retry
from timezonefinder import TimezoneFinder



In [ ]:
# Setup the Open-Meteo API client with cache and retry on error
cache_session = requests_cache.CachedSession('.cache', expire_after = 10)
retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
openmeteo = openmeteo_requests.Client(session = retry_session)

# Make sure all required weather variables are listed here
# The order of variables in hourly or daily is important to assign them correctly below
url = "https://api.open-meteo.com/v1/forecast"
lat, lng = -22.934, -43.180
tz = TimezoneFinder().certain_timezone_at(lat=lat, lng=lng)

params = {
	"latitude": lat,
	"longitude": lng,
	"hourly": ["temperature_2m", "precipitation_probability", "precipitation"],
    "forecast_hours" : 6,
	"current": ["temperature_2m", "precipitation", "wind_speed_10m", "wind_direction_10m"],
}
responses = openmeteo.weather_api(url, params=params)


In [ ]:
from numpy import round

def get_cardinal_wind_direction(degrees:float) -> str:
    directions = ['N ↓', 'NE ↙', 'E ←', 'SE ↖', 'S ↑', 'SW ↗', 'W →', 'NW ↘']
    idx = round(degrees / 45) % 8
    return directions[int(idx)]

def get_lat_lng_str(coordinates:float,type:str) -> str:
    if not type in ["lat", "lng"]:
        raise ValueError("Type must be 'lat' or 'lng'")
    if type == "lat":
        return f"{abs(coordinates)}°S" if coordinates < 0 else f"{coordinates}°N"
    else:
        return f"{abs(coordinates)}°W" if coordinates < 0 else f"{coordinates}°E"

In [ ]:
response = responses[0]
print(f"Coordinates: {get_lat_lng_str(response.Latitude(), 'lat')} {get_lat_lng_str(response.Longitude(), 'lng')}")
print(f"Elevation: {response.Elevation()} m asl")
print(f"Timezone: {tz}")

# Process current data. The order of variables needs to be the same as requested.
current = response.Current()
current_temperature_2m = current.Variables(0).Value()
current_precipitation = current.Variables(1).Value()
current_wind_speed_10m = current.Variables(2).Value()
current_wind_direction_10m = current.Variables(3).Value()

print(f"\nCurrent Weather Update: { pd.to_datetime(current.Time(), unit = 's', utc = True).tz_convert(tz) }")
print(f"Current temperature_2m: {int(round(current_temperature_2m, 1))} °C")
print(f"Current precipitation: {current_precipitation} mm")
print(f"Current wind_speed_10m: {current_wind_speed_10m} m/s")
print(f"Current wind_direction_10m: {get_cardinal_wind_direction(current_wind_direction_10m)} ({current_wind_direction_10m}°)")

# Process hourly data. The order of variables needs to be the same as requested.
hourly = response.Hourly()
hourly_temperature_2m = hourly.Variables(0).ValuesAsNumpy()
hourly_precipitation_probability = hourly.Variables(1).ValuesAsNumpy()
hourly_precipitation = hourly.Variables(2).ValuesAsNumpy()

hourly_data = {"date": pd.date_range(
	start = pd.to_datetime(hourly.Time(), unit = "s", utc = True).tz_convert(tz),
	end =  pd.to_datetime(hourly.TimeEnd(), unit = "s", utc = True).tz_convert(tz),
	freq = pd.Timedelta(seconds = hourly.Interval()),
	inclusive = "left"
)}

hourly_data["temperature_2m"] = hourly_temperature_2m
hourly_data["precipitation_probability"] = hourly_precipitation_probability
hourly_data["precipitation"] = hourly_precipitation

hourly_dataframe = pd.DataFrame(data = hourly_data)
print("\nHourly data\n", hourly_dataframe)
    

## HGBrasil API 

In [ ]:
import requests
from dotenv import load_dotenv
from os import getenv

load_dotenv()
api_key = getenv('HG_BRAZIL_API_KEY')
lat, lon = -22.934, -43.180

url = 'https://api.hgbrasil.com/weather'

params = {
    'lat': lat,
    'lon': lon,
    'key': api_key
}

response = requests.get(url, params=params)
data = response.json()

In [ ]:
data.keys()

In [ ]:

forecast = data.get("results", {}).get("forecast", [])
print(forecast)

## Meteoblue API 

In [ ]:
import requests
from dotenv import load_dotenv
from os import getenv


load_dotenv()

api_key = getenv('METEO_BLUE_API_KEY')
lat, lon = -22.934, -43.180
url = f"https://my.meteoblue.com/packages/current_basic-1h_basic-day?lat={lat}&lon={lon}&apikey={api_key}&forecast_days=1"

response = requests.get(url)
data = response.json()

### Saving request data to disk

In [ ]:
import json

with open("meteo_blue_response.json", "w") as f:
    json.dump(data, f, indent=4)

### Load saved data to avoid spending API call credits

In [ ]:
with open("meteo_blue_response.json", "r") as f:
    data = json.load(f)

In [ ]:
import pandas as pd

tz = data.get("metadata",{}).get("timezone_abbrevation","utc")
if tz.upper()=="UTC":
    tz = "UTC"
else:
    tz = f"Etc/{tz.replace("0","")}"

current = data.get("data_current", {})
predict = pd.DataFrame(data.get("data_1h", []))

In [ ]:
predict["winddirection_cardinal"] = predict["winddirection"].apply(get_cardinal_wind_direction)
predict = predict[[
    'time',
    'windspeed',
    'winddirection',
    'winddirection_cardinal',
    'temperature',
    'precipitation_probability',
    'precipitation',
]]
predict


# Pipeline Testing

In [ ]:
import pandas as pd
from psycopg import PostgresHook

In [ ]:



class RequestToPostgresOperator():
    """
    Custom operator to merge a pandas DataFrame into a Postgres table.

    Parameters
    ----------
    dataframe : pd.DataFrame
        The DataFrame containing data to merge.
    table : str
        Target Postgres table name.
    conn_id : str
        Airflow connection ID for Postgres.
    unique_key : str | Sequence[str]
        Column(s) that define uniqueness for conflict resolution.
    """

    def __init__(
        self,
        dataframe: pd.DataFrame,
        table: str,
        conn_id: str = "postgres_default",
        unique_key: str | Sequence[str] = "id",
        **kwargs: Any,
    ) -> None:
        super().__init__(**kwargs)
        self.dataframe = dataframe
        self.table = table
        self.conn_id = conn_id
        self.unique_key = unique_key

    def execute(self, context: Context) -> dict[str, Any]:
        hook = PostgresHook(postgres_conn_id=self.conn_id)
        conn = hook.get_conn()
        cursor = conn.cursor()

        inserted, updated = 0, 0

        try:
            cols = list(self.dataframe.columns)
            placeholders = ", ".join(["%s"] * len(cols))
            col_names = ", ".join(cols)

            # Build conflict clause
            if isinstance(self.unique_key, str):
                conflict_cols = self.unique_key
            else:
                conflict_cols = ", ".join(self.unique_key)

            update_clause = ", ".join([f"{c}=EXCLUDED.{c}" for c in cols if c not in self.unique_key])

            sql = f"""
                INSERT INTO {self.table} ({col_names})
                VALUES ({placeholders})
                ON CONFLICT ({conflict_cols})
                DO UPDATE SET {update_clause};
            """

            for row in self.dataframe.itertuples(index=False, name=None):
                cursor.execute(sql, row)
                if cursor.rowcount == 1:
                    inserted += 1
                else:
                    updated += 1

            conn.commit()
            result = MergeResult(inserted=inserted, updated=updated)
            self.log.info("Merge completed: %s", result.json())
            return result.dict()

        except Exception as e:
            conn.rollback()
            self.log.error("Error during merge: %s", str(e))
            raise
        finally:
            cursor.close()
            conn.close()